In [ ]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [ ]:
STYLE_MAP = {
    "Back": 0,
    "Front": 1,
    "Goblet": 2
}
NUM_STYLE = 3

POSTURE_MAP = {
    "_corr": 0,
    "_id": 1,
    "_rb": 2,
    "_toe": 3
}
NUM_POSTURE = 4


In [ ]:
DATA_ROOT = "..."

X = []
y_style = []
y_posture = []

for root, _, files in os.walk(DATA_ROOT):
    for file in files:
        if not file.endswith(".npy"):
            continue

        full_path = os.path.join(root, file)

        # posture label
        posture_label = None
        for suffix, pid in POSTURE_MAP.items():
            if suffix in root:
                posture_label = pid
                break

        # style label
        style_label = None
        for style, sid in STYLE_MAP.items():
            if f"/{style}/" in full_path:
                style_label = sid
                break

        if posture_label is None or style_label is None:
            continue

        data = np.load(full_path).reshape(50, 34)

        X.append(data)
        y_style.append(style_label)
        y_posture.append(posture_label)

X = np.array(X)
y_style = to_categorical(y_style, NUM_STYLE)
y_posture = to_categorical(y_posture, NUM_POSTURE)

print("X:", X.shape)
print("Style labels:", y_style.shape)
print("Posture labels:", y_posture.shape)


In [ ]:
X_train, X_val, y_style_train, y_style_val, y_posture_train, y_posture_val = train_test_split(
    X,
    y_style,
    y_posture,
    test_size=0.3,
    random_state=42,
    stratify=y_posture.argmax(axis=1)
)


In [ ]:
input_layer = Input(shape=(50, 34))

x = Conv1D(32, kernel_size=3, activation="relu")(input_layer)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
x = Dense(32, activation="relu")(x)

# Output 1: Squat style
out_style = Dense(NUM_STYLE, activation="softmax", name="style")(x)

# Output 2: Posture quality
out_posture = Dense(NUM_POSTURE, activation="softmax", name="posture")(x)

model_multi = Model(inputs=input_layer, outputs=[out_style, out_posture])


In [ ]:
model_multi.compile(
    optimizer="adam",
    loss={
        "style": "categorical_crossentropy",
        "posture": "categorical_crossentropy"
    },
    metrics={
        "style": "accuracy",
        "posture": "accuracy"
    }
)

model_multi.summary()


In [ ]:
history_multi = model_multi.fit(
    X_train,
    {"style": y_style_train, "posture": y_posture_train},
    validation_data=(
        X_val,
        {"style": y_style_val, "posture": y_posture_val}
    ),
    epochs=30,
    batch_size=16
)


In [ ]:
model_multi.save(
    "1D_CNN_multi_output.keras"
)


In [ ]:
import os
import numpy as np
from tensorflow.keras.utils import to_categorical

STYLE_MAP = {
    "Back": 0,
    "Front": 1,
    "Goblet": 2
}
NUM_STYLE = 3

POSTURE_MAP = {
    "_corr": 0,
    "_id": 1,
    "_rb": 2,
    "_toe": 3
}
NUM_POSTURE = 4

TEST_ROOT = "..."

X_test = []
y_style_test = []
y_posture_test = []

for root, _, files in os.walk(TEST_ROOT):
    for file in files:
        if not file.endswith(".npy"):
            continue

        full_path = os.path.join(root, file)

        # posture label
        posture_label = None
        for suffix, pid in POSTURE_MAP.items():
            if suffix in root:
                posture_label = pid
                break

        # style label
        style_label = None
        for style, sid in STYLE_MAP.items():
            if f"/{style}/" in full_path:
                style_label = sid
                break

        if posture_label is None or style_label is None:
            continue

        data = np.load(full_path).reshape(50, 34)

        X_test.append(data)
        y_style_test.append(style_label)
        y_posture_test.append(posture_label)

X_test = np.array(X_test)
y_style_test = to_categorical(y_style_test, NUM_STYLE)
y_posture_test = to_categorical(y_posture_test, NUM_POSTURE)

print("Test shapes:")
print(X_test.shape, y_style_test.shape, y_posture_test.shape)


In [ ]:
from tensorflow.keras.models import load_model

model_multi = load_model(
    "1D_CNN_multi_output.keras"
)


In [ ]:
test_results = model_multi.evaluate(
    X_test,
    {"style": y_style_test, "posture": y_posture_test},
    verbose=0
)

print("Test results:")
for name, value in zip(model_multi.metrics_names, test_results):
    print(f"{name}: {value:.4f}")


In [ ]:
from sklearn.metrics import classification_report

style_pred, posture_pred = model_multi.predict(X_test)

style_pred_cls = np.argmax(style_pred, axis=1)
posture_pred_cls = np.argmax(posture_pred, axis=1)

style_true_cls = np.argmax(y_style_test, axis=1)
posture_true_cls = np.argmax(y_posture_test, axis=1)


In [ ]:
print("STYLE classification report:")
print(classification_report(
    style_true_cls,
    style_pred_cls,
    target_names=["Back", "Front", "Goblet"]
))


In [ ]:
print("\nPOSTURE classification report:")
print(classification_report(
    posture_true_cls,
    posture_pred_cls,
    target_names=[
        "Correct",
        "Insufficient depth",
        "Rounded back",
        "Weight on toes"
    ]
))


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Style
cm_style = confusion_matrix(style_true_cls, style_pred_cls)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm_style,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Back", "Front", "Goblet"],
    yticklabels=["Back", "Front", "Goblet"]
)
plt.title("Style Confusion CNN Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Posture
cm_posture = confusion_matrix(posture_true_cls, posture_pred_cls)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_posture,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Corr", "ID", "RB", "Toe"],
    yticklabels=["Corr", "ID", "RB", "Toe"]
)
plt.title("Posture Confusion CNN Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
